### Import Libraries

In [1]:
import platform
from sqlalchemy import create_engine, text
import pandas as pd
from datetime import datetime, timedelta
pd.set_option('display.max_rows', None)

### Filtering Time Data

In [2]:
# Control Variables
cutoff_date = datetime.now()
N_Weeks = 4

# Find the Monday of the cutoff date's week
cutoff_week_monday = cutoff_date - timedelta(days = cutoff_date.weekday())

# Use the latest fully completed Monday-Friday week
if cutoff_date.weekday() >= 5:
    latest_complete_friday = cutoff_week_monday + timedelta(days=4)
else:
    latest_complete_friday = cutoff_week_monday - timedelta(days=3)

# Identify the start and end dates for the N-week analysis period
latest_complete_monday = latest_complete_friday - timedelta(days=4)
start_date = latest_complete_monday - timedelta(weeks=N_Weeks - 1)
end_date = latest_complete_friday

print(cutoff_date)
print(start_date)
print(end_date)


2026-05-28 00:13:25.958620
2026-04-27 00:13:25.958620
2026-05-22 00:13:25.958620


### Define Functions

In [3]:
#Establish environment for SQL connection
def db_connect (_driver = None):
    template = 'mssql+pyodbc:///?odbc_connect=DRIVER={};SERVER=dlyle.database.windows.net;DATABASE=DWV;UID=muesli;PWD=Viz(Data);'

    if not _driver is None:
        driver = _driver
    else:
        print('Guessing driver based on', platform.system(), platform.release(), platform.machine(), platform.platform())
        if platform.system() == 'Darwin': #MacOS
            if platform.machine() == 'arm64': #M1 chip
                driver = '/opt/homebrew/lib/libmsodbcsql.18.dylib'
            else:
                driver = '/Library/simba/sqlserverodbc/lib/libsqlserverodbc_sbu.dylib'
        else: #Windows and anything else
            driver = '{SQL Server}'

    c_str = template.format(driver)
    print(c_str)
    print('Attempting connection')   
    cxn = create_engine(c_str).connect()
    print('Success!')
    return cxn

### Aquire Data for Inventory

In [4]:
inventory_sql = f"""
WITH sub AS (                   
    SELECT 
        m.Material_Name,
        m.Material_Code,
        ic.Inventory_Date,
        ic.Location_Code,
        LAG(ic.closing_quantity) OVER (
            PARTITION BY ic.Material_Code, ic.Location_Code
            ORDER BY ic.Inventory_Date
        ) AS Opening_Quantity                       -- Use the previous closing inventory as the current day's opening inventory
    
    FROM Inventory_Counts AS ic
    
    INNER JOIN Materials AS m 
    ON ic.Material_Code = m.Material_Code
    
    WHERE ic.Location_Code IN ('02N', '02S', '02W') -- Keep only the three distribution centres
)  
                                                    -- Keep all data in the subquery to prevent date filter filtering out the first few lines' opening inventory

SELECT 
    sub.Material_Name,
    sub.Material_Code,
    sub.Inventory_Date,
    sub.Location_Code,
    sub.Opening_Quantity

FROM sub

WHERE sub.Inventory_Date BETWEEN '{start_date:%Y-%m-%d}' 
AND '{end_date:%Y-%m-%d}'                           -- Keep only data within the N-Weeks Period

ORDER BY 
    sub.Inventory_Date,
    sub.Material_Code,
    sub.Location_Code;
    """

cxn = db_connect()
dfinventory = pd.read_sql(text(inventory_sql), cxn, parse_dates = ['Inventory_Date'])
cxn.close()
display(dfinventory)
dfinventory.info()

Guessing driver based on Windows 11 AMD64 Windows-11-10.0.26200-SP0
mssql+pyodbc:///?odbc_connect=DRIVER={SQL Server};SERVER=dlyle.database.windows.net;DATABASE=DWV;UID=muesli;PWD=Viz(Data);
Attempting connection
Success!


,Material_Name,Material_Code,Inventory_Date,Location_Code,Opening_Quantity
0,500g Nut Muesli,BB-F01,2026-04-27,02N,1150.0
1,500g Nut Muesli,BB-F01,2026-04-27,02S,1150.0
2,500g Nut Muesli,BB-F01,2026-04-27,02W,1150.0
3,500g Blueberry Muesli,BB-F02,2026-04-27,02N,0.0
4,500g Blueberry Muesli,BB-F02,2026-04-27,02S,5201.0
...,...,...,...,...,...
715,1kg Original Muesli,BB-F15,2026-05-22,02S,0.0
716,1kg Original Muesli,BB-F15,2026-05-22,02W,0.0
717,1kg Mixed Fruit Muesli,BB-F16,2026-05-22,02N,0.0
718,1kg Mixed Fruit Muesli,BB-F16,2026-05-22,02S,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Material_Name     720 non-null    object        
 1   Material_Code     720 non-null    object        
 2   Inventory_Date    720 non-null    datetime64[ns]
 3   Location_Code     720 non-null    object        
 4   Opening_Quantity  720 non-null    float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 28.3+ KB


### Acquire Data for Sales

In [5]:
sales_sql = f"""
SELECT 
    so.Document_Number,
    so.Delivery_Date, 
    m.Material_Name,
    m.Material_Code,
    soi.Quantity,  
    c.Region_Code, 
    
    CASE                                                           
        WHEN c.Region_Code IN ('05', '07', '10') THEN '02W'
        WHEN c.Region_Code IN ('06', '08', '09', '16') THEN '02S'
        ELSE '02N'
    END AS distribution_centre_code    -- Map customer region to the distribution centre that fulfilled the sale

FROM Sales_Order_Items AS soi

INNER JOIN Sales_Orders AS so
    ON soi.Document_Number = so.Document_Number

INNER JOIN Customers AS c
    ON so.Customer_ID = c.Customer_ID

INNER JOIN Materials AS m
    ON soi.Material_Code = m.Material_Code

WHERE so.Delivery_Date Between '{start_date:%Y-%m-%d}' 
AND '{end_date:%Y-%m-%d}'                -- Keep only sales transactions within the N-Weeks period

ORDER BY
    so.Delivery_Date,
    soi.Material_Code,
    distribution_centre_code;
"""

cxn = db_connect()
dfsales = pd.read_sql(text(sales_sql), cxn, parse_dates = ["Delivery_Date"])
cxn.close()
display(dfsales)
dfsales.info()

Guessing driver based on Windows 11 AMD64 Windows-11-10.0.26200-SP0
mssql+pyodbc:///?odbc_connect=DRIVER={SQL Server};SERVER=dlyle.database.windows.net;DATABASE=DWV;UID=muesli;PWD=Viz(Data);
Attempting connection
Success!


,Document_Number,Delivery_Date,Material_Name,Material_Code,Quantity,Region_Code,distribution_centre_code
0,0000005042,2026-04-27,500g Nut Muesli,BB-F01,1150.0,03,02N
1,0000005056,2026-04-27,500g Blueberry Muesli,BB-F02,4402.0,06,02S
2,0000005047,2026-04-27,500g Blueberry Muesli,BB-F02,2416.0,05,02W
3,0000005047,2026-04-27,1kg Strawberry Muesli,BB-F13,4183.0,05,02W
4,0000005042,2026-04-27,1kg Mixed Fruit Muesli,BB-F16,4625.0,03,02N
...,...,...,...,...,...,...,...
101,0000005453,2026-05-20,1kg Mixed Fruit Muesli,BB-F16,2360.0,09,02S
102,0000005489,2026-05-21,500g Blueberry Muesli,BB-F02,3692.0,11,02N
103,0000005478,2026-05-21,500g Original Muesli,BB-F05,4788.0,09,02S
104,0000005471,2026-05-21,1kg Mixed Fruit Muesli,BB-F16,3645.0,12,02N


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Document_Number           106 non-null    object        
 1   Delivery_Date             106 non-null    datetime64[ns]
 2   Material_Name             106 non-null    object        
 3   Material_Code             106 non-null    object        
 4   Quantity                  106 non-null    float64       
 5   Region_Code               106 non-null    object        
 6   distribution_centre_code  106 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 5.9+ KB


##  Minimum stock level required to enable at least one sale 

In [6]:
# Calculate the median transaction quantity for each product, regardless of location

dfthreshold = (
    dfsales         # Select from sales data
    .groupby(["Material_Code", "Material_Name"]) #Group by Material Code and Name
    .Quantity.median()  #Calculate median of quantity
    .reset_index()   #Change grouped fields into normal columns
)

dfthreshold = dfthreshold.rename(   #rename column header for quantity as minimum threshold
    columns={
        "Quantity": "Minimum_Threshold"
    }
)

display(dfthreshold)
dfthreshold.info()

,Material_Code,Material_Name,Minimum_Threshold
0,BB-F01,500g Nut Muesli,4159.0
1,BB-F02,500g Blueberry Muesli,3761.0
2,BB-F05,500g Original Muesli,4847.0
3,BB-F13,1kg Strawberry Muesli,3004.0
4,BB-F14,1kg Raisin Muesli,3171.0
5,BB-F16,1kg Mixed Fruit Muesli,3285.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Material_Code      6 non-null      object 
 1   Material_Name      6 non-null      object 
 2   Minimum_Threshold  6 non-null      float64
dtypes: float64(1), object(2)
memory usage: 276.0+ bytes


##  Complete daily sales series 

In [7]:
#Aggregate actual sales transactions to daily level
Daily_sales = (dfsales               
               .groupby(['Material_Code',
                         'Material_Name',
                         'distribution_centre_code',
                         'Delivery_Date'
                        ])
               .Quantity.sum()
               .reset_index()
               .rename(columns={"Quantity": "Daily_Sales"}
              ))



# Use the sales table to identify products that are actually for sale in this period
dfproducts = (dfsales[[ "Material_Code","Material_Name"]]
    .drop_duplicates())


#Compute Distribution centres for creating a complete sales table base
dflocations = pd.DataFrame({
    "distribution_centre_code": ["02N", "02S", "02W"]
})


#Create all Monday–Friday dates in N-Weeks
dfdates = pd.DataFrame({
    "Delivery_Date": pd.date_range(
        start=start_date.date(),
        end=end_date.date(),
        freq="B")
})
#create the complete sales table base
Daily_sales_base = (
    dfproducts
    .merge(dflocations, how="cross")
    .merge(dfdates, how="cross")
)

#Merge actual sales and fill missing sales with 0
Daily_sales = Daily_sales_base.merge(
              Daily_sales,
              on=["Material_Code",
                  "Material_Name",
                  "distribution_centre_code",
                  "Delivery_Date"
                 ], how = "left")
Daily_sales["Daily_Sales"] = ( Daily_sales["Daily_Sales"].fillna(0))

# Turn the  daily sales table into a  Series

Daily_sales_series = (
    Daily_sales.set_index([
            "Material_Code",
            "Material_Name",
            "distribution_centre_code",
            "Delivery_Date"
        ])["Daily_Sales"]
)

display(Daily_sales_series)

Material_Code  Material_Name      distribution_centre_code  Delivery_Date
BB-F01         500g Nut Muesli    02N                       2026-04-27       1150.0
                                                            2026-04-28          0.0
                                                            2026-04-29       9000.0
                                                            2026-04-30          0.0
                                                            2026-05-01          0.0
                                                                              ...  
BB-F14         1kg Raisin Muesli  02W                       2026-05-18       3023.0
                                                            2026-05-19          0.0
                                                            2026-05-20          0.0
                                                            2026-05-21          0.0
                                                            2026-05-22          0.0
Na

## Low Inventory Bias Adjustment

In [8]:
# Match Daily sales with opening quantity

Daily_sales_with_opening_quantity = Daily_sales.merge(
    dfinventory[
    [
        "Material_Code",
        "Inventory_Date",
        "Location_Code",
        "Opening_Quantity"
    ]
],
    left_on=[
        "Material_Code",
        "Delivery_Date",
        "distribution_centre_code"
    ],
    right_on=[
        "Material_Code",
        "Inventory_Date",
        "Location_Code"
    ],
    how="left"
)

#remove duplicate columns for clarity

Daily_sales_with_opening_quantity = Daily_sales_with_opening_quantity.drop(
    columns=["Inventory_Date", "Location_Code"]
)

#match minimum threshold with the data for comparison

complete_daily_sales = Daily_sales_with_opening_quantity.merge(
    dfthreshold,
    on=['Material_Code',
        'Material_Name'],
    how = "left")

#Only keep a copy of transactions that satisfy the minimum threshold

adjusted_daily_sales = complete_daily_sales[
    complete_daily_sales["Opening_Quantity"] >= 
    complete_daily_sales["Minimum_Threshold"]].copy().reset_index()
adjusted_daily_sales

,index,Material_Code,Material_Name,distribution_centre_code,Delivery_Date,Daily_Sales,Opening_Quantity,Minimum_Threshold
0,2,BB-F01,500g Nut Muesli,02N,2026-04-29,9000.0,9000.0,4159.0
1,17,BB-F01,500g Nut Muesli,02N,2026-05-20,4159.0,4242.0,4159.0
2,22,BB-F01,500g Nut Muesli,02S,2026-04-29,9000.0,9000.0,4159.0
3,37,BB-F01,500g Nut Muesli,02S,2026-05-20,4242.0,4242.0,4159.0
4,42,BB-F01,500g Nut Muesli,02W,2026-04-29,7588.0,9000.0,4159.0
...,...,...,...,...,...,...,...,...
127,335,BB-F14,1kg Raisin Muesli,02S,2026-05-18,3170.0,9012.0,3171.0
128,336,BB-F14,1kg Raisin Muesli,02S,2026-05-19,0.0,5842.0,3171.0
129,337,BB-F14,1kg Raisin Muesli,02S,2026-05-20,5842.0,5842.0,3171.0
130,351,BB-F14,1kg Raisin Muesli,02W,2026-05-12,6689.0,8250.0,3171.0


## Average daily sales 

In [9]:
#Compute product-location daily average sales
avg_daily_sales = (adjusted_daily_sales
    .groupby(['Material_Code',
             'Material_Name',
             'distribution_centre_code'
            ])
    .Daily_Sales
    .mean()
    .reset_index()
    .rename(columns = {'Daily_Sales':'Avg_Daily_Sales'})
                  )
avg_daily_sales

,Material_Code,Material_Name,distribution_centre_code,Avg_Daily_Sales
0,BB-F01,500g Nut Muesli,02N,6579.500000
1,BB-F01,500g Nut Muesli,02S,6621.000000
2,BB-F01,500g Nut Muesli,02W,1068.909091
3,BB-F02,500g Blueberry Muesli,02N,5733.000000
4,BB-F02,500g Blueberry Muesli,02S,4948.250000
...,...,...,...,...
13,BB-F14,1kg Raisin Muesli,02S,2845.166667
14,BB-F14,1kg Raisin Muesli,02W,4887.000000
15,BB-F16,1kg Mixed Fruit Muesli,02N,4576.285714
16,BB-F16,1kg Mixed Fruit Muesli,02S,6289.750000


## Adjusting for Warehouse Demand

In [10]:
#compute the net sales quantity demand for the main warehouse
warehouse_demand = (avg_daily_sales
    .groupby(['Material_Code',
              'Material_Name'])
    .Avg_Daily_Sales
    .sum()
    .reset_index()
                   )
# Add the column for main factory warehouse location code
warehouse_demand["distribution_centre_code"] = "02"

# Reorder columns to match avg_daily_sales
warehouse_demand = warehouse_demand[
    [ "Material_Code",
        "Material_Name",
        "distribution_centre_code",
        "Avg_Daily_Sales"]
]

# Combine distribution centre demand and factory warehouse demand into one final table

final_demand = pd.concat(
    [avg_daily_sales, warehouse_demand],
     ignore_index = True
    )

final_demand

,Material_Code,Material_Name,distribution_centre_code,Avg_Daily_Sales
0,BB-F01,500g Nut Muesli,02N,6579.500000
1,BB-F01,500g Nut Muesli,02S,6621.000000
2,BB-F01,500g Nut Muesli,02W,1068.909091
3,BB-F02,500g Blueberry Muesli,02N,5733.000000
4,BB-F02,500g Blueberry Muesli,02S,4948.250000
...,...,...,...,...
19,BB-F02,500g Blueberry Muesli,02,12407.138889
20,BB-F05,500g Original Muesli,02,12380.671429
21,BB-F13,1kg Strawberry Muesli,02,3110.278788
22,BB-F14,1kg Raisin Muesli,02,13443.166667
